In [2]:
import pandas as pd
import requests
from datetime import datetime


In [3]:
cities = {
    "Freiburg":  {"lat": 47.99, "lon": 7.84},
    "Stuttgart": {"lat": 48.78, "lon": 9.18},
    "Karlsruhe": {"lat": 49.00, "lon": 8.40}
}

print("Cities defined:")
for city in cities:
    print(f"   → {city}: lat={cities[city]['lat']}, lon={cities[city]['lon']}")

Cities defined:
   → Freiburg: lat=47.99, lon=7.84
   → Stuttgart: lat=48.78, lon=9.18
   → Karlsruhe: lat=49.0, lon=8.4


In [9]:
# Cell 3 - Fetch live data from Open-Meteo API for all cities
all_data = []  # empty list to collect data from all cities

for city, coords in cities.items():
    
    # The API web address
    url = "https://api.open-meteo.com/v1/forecast"
    
    # What we are asking the API to give us
    params = {
        "latitude":  coords["lat"],
        "longitude": coords["lon"],
        "hourly": "temperature_2m,windspeed_10m,shortwave_radiation",
        "forecast_days": 3,
        "timezone": "Europe/Berlin"
    }
    
    # calling the API
    response = requests.get(url, params=params)
    
    # Converting the response to readable Python data
    data = response.json()

    
    # Organizing into a data frame
    df = pd.DataFrame({
        "city":                 city,
        "timestamp":            data["hourly"]["time"],
        "temperature_c":        data["hourly"]["temperature_2m"],
        "windspeed_kmh":        data["hourly"]["windspeed_10m"],
        "solar_radiation_wm2":  data["hourly"]["shortwave_radiation"]
    })

    all_data.append(df)
    print(f"{city}: {len(df)} rows fetched")

print("\nAll cities fetched successfully!")

Freiburg: 72 rows fetched
Stuttgart: 72 rows fetched
Karlsruhe: 72 rows fetched

All cities fetched successfully!


In [11]:
#combining all three cities data
df_combined_data=pd.concat(all_data,ignore_index=True)

#Converting time stamp from text to real format
df_combined_data["timestamp"]=pd.to_datetime(df_combined_data["timestamp"])

#adding a column to record exact data fetch time
df_combined_data["fetched_at"]=datetime.now()

print(f"Total rows : {len(df_combined_data)}")
print(f"Columns: {df_combined_data.columns.tolist()}")
df_combined_data.head()

Total rows : 216
Columns: ['city', 'timestamp', 'temperature_c', 'windspeed_kmh', 'solar_radiation_wm2', 'fetched_at']


,city,timestamp,temperature_c,windspeed_kmh,solar_radiation_wm2,fetched_at
0,Freiburg,2026-04-03 00:00:00,5.8,7.8,0.0,2026-04-03 13:34:50.437383
1,Freiburg,2026-04-03 01:00:00,4.8,5.7,0.0,2026-04-03 13:34:50.437383
2,Freiburg,2026-04-03 02:00:00,4.4,7.5,0.0,2026-04-03 13:34:50.437383
3,Freiburg,2026-04-03 03:00:00,4.5,6.8,0.0,2026-04-03 13:34:50.437383
4,Freiburg,2026-04-03 04:00:00,3.8,5.9,0.0,2026-04-03 13:34:50.437383


In [12]:
#Classification of Wind Energy potential
df_combined_data["wind_potential"]=pd.cut(
    df_combined_data["windspeed_kmh"],
    bins=[0,10,20,35,100],
    labels=["Low","Medium","High","Very High"]
)

#Classification of Solar Energy potential
df_combined_data["solar_potential"]=pd.cut(
    df_combined_data["solar_radiation_wm2"],
    bins=[-1,50,200,500,1000],
    labels=["Low","Medium","High","Very High"]
)

#Peak Energy Demand Signal
df_combined_data["demand_signal"]=df_combined_data["temperature_c"].apply(
    lambda t: "Peak Demand" if t>28 or t<2 else "Normal Demand"
)

# 4. Extract time features for Power BI slicers which makes filters and slicers ready to use and minimize use of DAX
df_combined_data["hour"]=df_combined_data["timestamp"].dt.hour
df_combined_data["date"]=df_combined_data["timestamp"].dt.date
df_combined_data["day"]=df_combined_data["timestamp"].dt.day_name()

print(df_combined_data.columns.to_list())

['city', 'timestamp', 'temperature_c', 'windspeed_kmh', 'solar_radiation_wm2', 'fetched_at', 'wind_potential', 'solar_potential', 'demand_signal', 'hour', 'date', 'day']


In [15]:
#Connect to PostgreSQL and save data
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Connection details
username = "postgres"
password = quote_plus("Radhasyam@19")
host     = "localhost"
port     = "5432"
database = "energy_dashboard"  

# Create engine
engine = create_engine(f"postgresql://{username}:{password}@{host}:{port}/{database}")

# Save to PostgreSQL
df_combined_data.to_sql(
    name="energy_data",   # new table name
    con=engine,
    if_exists="append",
    index=False
)

print(f"{len(df_combined_data)} rows saved to PostgreSQL successfully!")
print(f"Table name: energy_data")
print(f"Database: energy_dashboard")

216 rows saved to PostgreSQL successfully!
Table name: energy_data
Database: energy_dashboard


In [16]:
#store the data frame as a csv
df_combined_data.to_csv("../data/energy_latest.csv",index=False)
print("CSV exported Successfully")

CSV exported Successfully
